# **Task 3: Predictions on three data sets A, B, C**.

This notebook validates candidate models using training data only. It checks each train/public-test pair, uses expanding chronological validation, selects a model separately for A, B, and C, refits using the full training file, and exports predictions in the public test `t` order. Public test features are used only for final predictions.

In [12]:
import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config
import src.models
importlib.reload(src.config)
importlib.reload(src.models)
import src
importlib.reload(src)

from src import (
    TASK3_OUTER_TRAIN_FRACTIONS,
    TASK3_INNER_VALIDATION_FRACTION,
    TASK3_RIDGE_Z_GRID,
    fit_task3_model_with_validation,
)

DATA_DIR = PROJECT_ROOT / 'datasets'
PREDICTION_DIR = PROJECT_ROOT / 'predictions'
STUDENT_ID = 'A0352819W'

## 1. Validate and align each data pair

Training feature order defines the model input order. The loader rejects missing/extra features, nonnumeric or nonfinite data, malformed `feature1..featureP` names, invalid `t`, and overlap between train and public-test indices. If the public test columns are permuted, they are reordered to match training.

In [13]:
def _validated_integer_index(values, name):
    numeric = pd.to_numeric(values, errors='coerce').to_numpy(dtype=float)
    if not np.isfinite(numeric).all() or not np.equal(numeric, np.floor(numeric)).all():
        raise ValueError(f'{name} must contain finite integer values')
    index = numeric.astype(np.int64)
    if len(np.unique(index)) != len(index) or np.any(np.diff(index) <= 0):
        raise ValueError(f'{name} must be strictly increasing with no duplicates')
    return index


def load_task3_pair(letter):
    if letter not in {'A', 'B', 'C'}:
        raise ValueError('letter must be A, B, or C')
    train = pd.read_csv(DATA_DIR / f'pair{letter}_train.csv')
    public = pd.read_csv(DATA_DIR / f'pair{letter}_test_features.csv')

    feature_columns = [column for column in train.columns if column.startswith('feature')]
    expected_features = [f'feature{i}' for i in range(1, len(feature_columns) + 1)]
    expected_train_columns = ['t', *expected_features, 'return']
    if list(train.columns) != expected_train_columns:
        raise ValueError(f'{letter}: training columns must be t, feature1..featureP, return in order')
    if set(public.columns) != {'t', *expected_features} or len(public.columns) != len(expected_features) + 1:
        raise ValueError(f'{letter}: public test features do not match training features')

    train_t = _validated_integer_index(train['t'], f'{letter} training t')
    test_t = _validated_integer_index(public['t'], f'{letter} public-test t')
    if set(train_t).intersection(test_t):
        raise ValueError(f'{letter}: train and public-test t values overlap')

    X_train = train[expected_features].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    y_train = pd.to_numeric(train['return'], errors='coerce').to_numpy(dtype=float)
    X_test = public[expected_features].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    if not np.isfinite(X_train).all() or not np.isfinite(y_train).all() or not np.isfinite(X_test).all():
        raise ValueError(f'{letter}: features and labels must be numeric and finite')

    return X_train, y_train, X_test, test_t, expected_features


pair_data = {letter: load_task3_pair(letter) for letter in 'ABC'}
display(pd.DataFrame([
    {'dataset': letter, 'n_train': len(values[1]), 'n_test': len(values[3]), 'n_features': values[0].shape[1]}
    for letter, values in pair_data.items()
]))

,dataset,n_train,n_test,n_features
0,A,600,1000,600
1,B,240,4000,2400
2,C,360,2000,1800


## 2. Training-only expanding validation

All eight candidates (zero, training mean, no-intercept Ridge, no-intercept Lasso, intercept Ridge, Elastic Net, PCR, and PLS) use the same three expanding outer validation blocks: 60–70%, 70–80%, and 80–100% of the ordered training rows. Tunable models select hyperparameters only inside each outer training prefix, using its final 20% as an inner validation block, then refit on that prefix. The outer block is used only for evaluation.

Metrics are pooled over all outer predictions: validation MSE, paper OOS $R^2$, expected timing return, uncentered Sharpe, and variance-based Sharpe. Model selection first excludes candidates whose pooled MSE is worse than the zero forecast; among eligible candidates, it ranks by Sharpe, then expected timing return, then paper $R^2$. Per-fold scores are displayed as a stability diagnostic because pooled Sharpe can be noisy with short validation blocks.

In [17]:
MODEL_NAMES = ('zero', 'mean', 'ridge', 'lasso', 'ridge_intercept', 'elastic_net', 'pcr', 'pls')


def score_predictions(y_true, y_pred):
    timing_returns = y_true * y_pred
    mse = float(np.mean((y_true - y_pred) ** 2))
    denominator = float(np.mean(y_true ** 2))
    exp_return = float(np.mean(timing_returns))
    second_moment = float(np.mean(timing_returns ** 2))
    variance = float(np.var(timing_returns))
    return {
        'validation_mse': mse,
        'r2_paper': 1.0 - mse / denominator if denominator > 1e-12 else 0.0,
        'exp_return': exp_return,
        'sharpe': exp_return / np.sqrt(second_moment) if second_moment > 1e-12 else 0.0,
        'sharpe_var': exp_return / np.sqrt(variance) if variance > 1e-12 else 0.0,
    }


def expanding_validation(X, y):
    n_rows = len(y)
    train_ends = [int(fraction * n_rows) for fraction in TASK3_OUTER_TRAIN_FRACTIONS]
    validation_ends = train_ends[1:] + [n_rows]
    outer_fold_predictions = []
    targets = []
    tuning_records = []

    for fold, (train_end, validation_end) in enumerate(zip(train_ends, validation_ends), start=1):
        X_prefix, y_prefix = X[:train_end], y[:train_end]
        X_valid, y_valid = X[train_end:validation_end], y[train_end:validation_end]
        fold_predictions = {
            'zero': np.zeros(len(y_valid)),
            'mean': np.full(len(y_valid), y_prefix.mean()),
        }
        for model_name in MODEL_NAMES[2:]:
            estimator, metadata = fit_task3_model_with_validation(
                model_name,
                X_prefix,
                y_prefix,
                validation_fraction=TASK3_INNER_VALIDATION_FRACTION,
            )
            fold_predictions[model_name] = np.asarray(estimator.predict(X_valid)).reshape(-1)
            tuning_records.append({'fold': fold, 'n_inner_train': train_end - int(TASK3_INNER_VALIDATION_FRACTION * train_end), 'model': model_name, **metadata})
        outer_fold_predictions.append(fold_predictions)
        targets.append(y_valid)

    y_oof = np.concatenate(targets)
    pooled_predictions = {
        model_name: np.concatenate([fold[model_name] for fold in outer_fold_predictions])
        for model_name in MODEL_NAMES
    }
    scores = {
        model_name: score_predictions(y_oof, predictions)
        for model_name, predictions in pooled_predictions.items()
    }
    fold_scores = []
    for fold_index, fold_predictions in enumerate(outer_fold_predictions, start=1):
        for model_name, predictions in fold_predictions.items():
            fold_scores.append({
                'fold': fold_index,
                'model': model_name,
                **score_predictions(targets[fold_index - 1], predictions),
            })

    zero_mse = scores['zero']['validation_mse']
    mse_tolerance = max(1e-12, 1e-9 * zero_mse)
    eligible_models = [
        name for name in MODEL_NAMES
        if scores[name]['validation_mse'] <= zero_mse + mse_tolerance
    ]
    selected_model = max(
        eligible_models,
        key=lambda name: (
            scores[name]['sharpe'],
            scores[name]['exp_return'],
            scores[name]['r2_paper'],
        ),
    )
    return {
        'scores': scores,
        'fold_scores': fold_scores,
        'tuning': tuning_records,
        'eligible_models': eligible_models,
        'selected_model': selected_model,
    }


validation_results = {}
for letter, (X_train, y_train, _, _, _) in pair_data.items():
    result = expanding_validation(X_train, y_train)
    validation_results[letter] = result
    print(f'{letter}: selected {result["selected_model"]}; eligible={result["eligible_models"]}')
    display(pd.DataFrame(result['scores']).T.sort_values(['sharpe', 'exp_return', 'r2_paper'], ascending=False))
    display(pd.DataFrame(result['fold_scores']))
    display(pd.DataFrame(result['tuning']))

A: selected ridge; eligible=['zero', 'ridge', 'ridge_intercept']


,validation_mse,r2_paper,exp_return,sharpe,sharpe_var
ridge,1.166759,0.034256,0.045086,0.179527,0.182492
ridge_intercept,1.169520,0.031971,0.045170,0.177658,0.180530
pls,1.344101,-0.112533,0.108616,0.162226,0.164404
mean,1.208181,-0.000029,0.001058,0.020720,0.020724
lasso,1.217917,-0.008088,0.002572,0.018201,0.018204
elastic_net,1.213921,-0.004780,0.000952,0.010196,0.010196
pcr,1.256276,-0.039838,0.000052,0.000270,0.000270
zero,1.208146,0.000000,0.000000,0.000000,0.000000


,fold,model,validation_mse,r2_paper,exp_return,sharpe,sharpe_var
0,1,zero,1.234361,0.000000,0.000000,0.000000,0.000000
1,1,mean,1.231035,0.002694,0.002553,0.054461,0.054541
2,1,ridge,1.207309,0.021915,0.041523,0.164774,0.167057
3,1,lasso,1.220449,0.011270,0.015038,0.107736,0.108367
4,1,ridge_intercept,1.207125,0.022065,0.042161,0.171981,0.174582
5,1,elastic_net,1.224063,0.008343,0.007638,0.107440,0.108066
6,1,pcr,1.347283,-0.091483,-0.014808,-0.055424,-0.055509
7,1,pls,1.427695,-0.156627,0.083561,0.136840,0.138140
8,2,zero,1.070325,0.000000,0.000000,0.000000,0.000000
9,2,mean,1.065250,0.004742,0.003542,0.076398,0.076622


,fold,n_inner_train,model,z,inner_validation_mse,n_candidates,alpha,l1_ratio,n_components
0,1,288,ridge,5.0,1.140983,16,NaN,NaN,NaN
1,1,288,lasso,NaN,1.152180,16,0.157828,NaN,NaN
2,1,288,ridge_intercept,5.0,1.161736,16,NaN,NaN,NaN
3,1,288,elastic_net,NaN,1.175660,36,0.230541,0.9,NaN
4,1,288,pcr,NaN,1.119286,9,NaN,NaN,60.0
5,1,288,pls,NaN,1.260533,9,NaN,NaN,1.0
6,2,336,ridge,5.0,1.197619,16,NaN,NaN,NaN
7,2,336,lasso,NaN,1.220616,16,0.141042,NaN,NaN
8,2,336,ridge_intercept,5.0,1.199545,16,NaN,NaN,NaN
9,2,336,elastic_net,NaN,1.216613,36,0.206982,0.9,NaN


B: selected zero; eligible=['zero', 'lasso']


...

,fold,n_inner_train,model,z,inner_validation_mse,n_candidates,alpha,l1_ratio,n_components
0,1,116,ridge,10000.0,1.146969,16,NaN,NaN,NaN
1,1,116,lasso,NaN,1.146816,16,0.369759,NaN,NaN
2,1,116,ridge_intercept,10000.0,1.202825,16,NaN,NaN,NaN
3,1,116,elastic_net,NaN,1.202602,36,3.729636,0.1,NaN
4,1,116,pcr,NaN,1.190107,9,NaN,NaN,20.0
5,1,116,pls,NaN,1.342779,9,NaN,NaN,1.0
6,2,135,ridge,10000.0,0.802391,16,NaN,NaN,NaN
7,2,135,lasso,NaN,0.802275,16,0.372006,NaN,NaN
8,2,135,ridge_intercept,10000.0,0.809896,16,NaN,NaN,NaN
9,2,135,elastic_net,NaN,0.809808,36,3.641538,0.1,NaN


C: selected mean; eligible=['zero', 'mean', 'lasso', 'elastic_net']


,validation_mse,r2_paper,exp_return,sharpe,sharpe_var
mean,1.303163,0.003984,0.006014,0.065224,0.065363
elastic_net,1.303163,0.003984,0.006014,0.065224,0.065363
pcr,1.314377,-0.004587,0.006372,0.042475,0.042514
pls,1.458992,-0.115117,0.007858,0.017676,0.017679
ridge_intercept,1.328842,-0.015643,0.000801,0.004828,0.004828
zero,1.308376,0.000000,0.000000,0.000000,0.000000
lasso,1.308376,0.000000,0.000000,0.000000,0.000000
ridge,1.327697,-0.014767,-0.006410,-0.075828,-0.076047


,fold,model,validation_mse,r2_paper,exp_return,sharpe,sharpe_var
0,1,zero,1.373168,0.000000,0.000000,0.000000,0.000000
1,1,mean,1.364522,0.006296,0.006399,0.084744,0.085050
2,1,ridge,1.372978,0.000139,0.000095,0.266044,0.275991
3,1,lasso,1.373168,0.000000,0.000000,0.000000,0.000000
4,1,ridge_intercept,1.364318,0.006445,0.006497,0.086108,0.086429
5,1,elastic_net,1.364522,0.006296,0.006399,0.084744,0.085050
6,1,pcr,1.340229,0.023988,0.018747,0.223622,0.229432
7,1,pls,1.284401,0.064644,0.100684,0.265401,0.275273
8,2,zero,1.610122,0.000000,0.000000,0.000000,0.000000
9,2,mean,1.576917,0.020623,0.019004,0.216117,0.221348


,fold,n_inner_train,model,z,inner_validation_mse,n_candidates,alpha,l1_ratio,n_components
0,1,173,ridge,10000.0,1.512134,16,NaN,NaN,NaN
1,1,173,lasso,NaN,1.512076,16,0.289692,NaN,NaN
2,1,173,ridge_intercept,10000.0,1.528205,16,NaN,NaN,NaN
3,1,173,elastic_net,NaN,1.528133,36,2.972760,0.1,NaN
4,1,173,pcr,NaN,1.513950,9,NaN,NaN,1.0
5,1,173,pls,NaN,1.620300,9,NaN,NaN,2.0
6,2,201,ridge,25.0,1.555090,16,NaN,NaN,NaN
7,2,201,lasso,NaN,1.565003,16,0.282365,NaN,NaN
8,2,201,ridge_intercept,10.0,1.542091,16,NaN,NaN,NaN
9,2,201,elastic_net,NaN,1.557504,36,2.900660,0.1,NaN


## 3. Refit and predict

Use the selected model family from the eight-model comparison. Tunable models perform a final chronological 80/20 alpha/complexity selection using only the complete training file, then refit on all training rows. Zero and training-mean baselines require no tuning. Public test labels remain unavailable and are never used.

In [18]:
def fit_selected_model(letter):
    X_train, y_train, X_test, test_t, feature_columns = pair_data[letter]
    model_name = validation_results[letter]['selected_model']
    if model_name == 'zero':
        yhat = np.zeros(len(test_t))
        metadata = {'model': model_name}
    elif model_name == 'mean':
        yhat = np.full(len(test_t), y_train.mean())
        metadata = {'model': model_name, 'training_mean': float(y_train.mean())}
    else:
        estimator, metadata = fit_task3_model_with_validation(
            model_name,
            X_train,
            y_train,
            validation_fraction=TASK3_INNER_VALIDATION_FRACTION,
        )
        yhat = np.asarray(estimator.predict(X_test)).reshape(-1)
        metadata = {'model': model_name, **metadata}

    if yhat.shape != (len(test_t),) or not np.isfinite(yhat).all():
        raise RuntimeError(f'{letter}: invalid predictions')
    return test_t, yhat, metadata


prediction_results = {letter: fit_selected_model(letter) for letter in 'ABC'}
display(pd.DataFrame([
    {'dataset': letter, **prediction_results[letter][2], 'n_predictions': len(prediction_results[letter][1])}
    for letter in 'ABC'
]))

,dataset,model,z,inner_validation_mse,n_candidates,n_predictions,training_mean
0,A,ridge,5.0,1.211927,16.0,1000,NaN
1,B,zero,NaN,NaN,NaN,4000,NaN
2,C,mean,NaN,NaN,NaN,2000,-0.074727


## 4. Export submission files

Set `STUDENT_ID` in the setup cell to your actual student ID before running this cell. The output has exactly two columns (`t`, `yhat`), is UTF-8 encoded, and preserves public-test row order.

In [19]:
if not STUDENT_ID or STUDENT_ID == 'YOUR_STUDENT_ID' or any(char in STUDENT_ID for char in '/\\'):
    raise ValueError('Set STUDENT_ID to your actual ID before exporting predictions')

PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
export_paths = {}
for letter in 'ABC':
    test_t, yhat, _ = prediction_results[letter]
    path = PREDICTION_DIR / f'{STUDENT_ID}_predictions_{letter}.csv'
    pd.DataFrame({'t': test_t, 'yhat': yhat}).to_csv(path, index=False, encoding='utf-8')
    saved = pd.read_csv(path)
    if list(saved.columns) != ['t', 'yhat'] or not np.array_equal(saved['t'].to_numpy(), test_t):
        raise RuntimeError(f'{letter}: exported prediction file failed schema/order verification')
    export_paths[letter] = path

print(export_paths)

{'A': WindowsPath('d:/Desktop/5205-project1/predictions/A0352819W_predictions_A.csv'), 'B': WindowsPath('d:/Desktop/5205-project1/predictions/A0352819W_predictions_B.csv'), 'C': WindowsPath('d:/Desktop/5205-project1/predictions/A0352819W_predictions_C.csv')}
